 # Chapter 2 – Sector Proxy vs Stock Representativeness

 This notebook:

 - Loads processed data and runs the Chapter 2 analysis for each sector.

 - Produces:

   - UMAP cluster plot (intra-sector structure),

   - Sector proxy vs representative stock (normalized index),

   - Tracking error comparison (single stock and Top-5 basket).

In [1]:
from pathlib import Path
import pandas as pd
import plotly.io as pio
from pathlib import Path

import chapter2_intra_sector_test as ch2

# Set Plotly renderer for your environment: "vscode" or "notebook"
pio.renderers.default = "vscode"  # change to "notebook" if preferred


 ## Load parquet data

In [2]:
data = ch2.load_data(Path("data_processed"))
df_stock_sect = data["stocks_sect"]
df_stock_sect.head()


,Date,Weekday,Open,High,Low,Close,Adj Close,Volume,Symbol,Volatility,...,Sector,Return,SectorVolume,Weight,RollingVol20,WeightedVol20,RollingVol60,WeightedVol60,RollingVol120,WeightedVol120
0,2005-01-03,0,17.238913,17.296137,16.809729,17.081545,14.690905,3378800.0,A,0.028216,...,Healthcare,NaN,2.060590e+08,0.016397,NaN,NaN,NaN,NaN,NaN,NaN
1,2005-01-04,1,17.010014,17.160229,16.487841,16.630901,14.303329,3746900.0,A,0.039529,...,Healthcare,-0.026382,2.227180e+08,0.016824,NaN,NaN,NaN,NaN,NaN,NaN
2,2005-01-05,2,16.573677,16.917025,16.537910,16.623749,14.297175,3898600.0,A,0.022874,...,Healthcare,-0.000430,2.248905e+08,0.017336,NaN,NaN,NaN,NaN,NaN,NaN
3,2005-01-06,3,16.738197,16.766809,16.230330,16.258942,13.983426,3158600.0,A,0.032051,...,Healthcare,-0.021945,1.718338e+08,0.018382,NaN,NaN,NaN,NaN,NaN,NaN
4,2005-01-07,4,16.223175,16.416309,16.187410,16.244635,13.971125,2624300.0,A,0.014109,...,Healthcare,-0.000880,1.730605e+08,0.015164,NaN,NaN,NaN,NaN,NaN,NaN


 ## List available sectors

In [3]:
sectors = sorted(df_stock_sect["Sector"].dropna().unique())
print(f"Found {len(sectors)} sectors:")
print(sectors)


Found 11 sectors:
['Basic Materials', 'Communication Services', 'Consumer Cyclical', 'Consumer Defensive', 'Energy', 'Financial Services', 'Healthcare', 'Industrials', 'Real Estate', 'Technology', 'Utilities']


 ## Helper to run a sector

In [4]:
def analyze_sector(df_stock_sect, sector_name, return_col="Return", window_rep=60, te_windows=(60, 120), min_overlap_corr=30, distance_scale=2.0):
    """Run sector analysis with pairwise-overlap correlations over full history."""
    return ch2.run_sector_analysis(
        df_stock_sect=df_stock_sect,
        sector=sector_name,
        return_col=return_col,
        window_rep=window_rep,
        te_windows=te_windows,
        min_overlap_corr=min_overlap_corr,
        distance_scale=distance_scale,
    )


 ## Single-sector analysis

 Set the sector you want to analyze.

In [5]:
SECTOR_NAME = "Healthcare"  # change as needed

results = analyze_sector(df_stock_sect, SECTOR_NAME, min_overlap_corr=30, distance_scale=2.0)

fig_umap = results["fig_umap"]
fig_index = results["fig_index"]
fig_te = results["fig_te"]

if fig_umap is not None:
    fig_umap.show()
fig_index.show()
fig_te.show()


/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


 ## Key tables and metrics

In [6]:
scores_df = results["scores"]
risk_df = results["risk"]
basket_tickers = results["top_tickers"]
single_ticker = results["single_ticker"]

print("Best representative stock:", single_ticker)
print("Top-5 basket:", basket_tickers)

display(scores_df.sort_values("rep_score", ascending=False).head(10))
if basket_tickers:
    display(risk_df.loc[basket_tickers])


Best representative stock: PFE
Top-5 basket: ['PFE', 'GILD', 'BSX', 'AMGN', 'TMO']


,rep_score,avg_corr,n_obs
Symbol,,,
PFE,0.445676,0.445676,3798
GILD,0.402742,0.402742,3798
BSX,0.401200,0.401200,3798
AMGN,0.393008,0.393008,3798
TMO,0.386719,0.386719,3798
EXEL,0.386638,0.386638,3798
MRK,0.380724,0.380724,3798
A,0.380480,0.380480,3798
LLY,0.377367,0.377367,3798


,vol,max_drawdown,es_alpha
PFE,0.014112,0.530159,-0.032263
GILD,0.019043,0.457398,-0.042487
BSX,0.021549,0.858767,-0.050564
AMGN,0.016737,0.536149,-0.036676
TMO,0.016500,0.539075,-0.038316


 ## Interpretation scaffold

 - **UMAP clusters:** Note clusters indicating similarity; highlight outliers or dense sub-groups. _[Add comments]_

 - **Index comparison:** Does the representative (and basket) track the sector proxy closely? Identify divergence periods. _[Add comments]_

 - **Tracking error:** Typical TE levels/spikes? Is the basket consistently lower TE than single stock across 60d/120d? _[Add comments]_

 ## (Optional) Loop over all sectors

 Set `SAVE_FIGS=True` to write HTML files for each plot.

In [7]:
SAVE_FIGS = True
FIG_DIR = Path("plots_ch2")
if SAVE_FIGS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

for sect in sectors:
    print(f"\n=== {sect} ===")
    res = analyze_sector(df_stock_sect, sect, min_overlap_corr=30, distance_scale=2.0)

    fu, fi, ft = res["fig_umap"], res["fig_index"], res["fig_te"]

    if fu is not None:
        fu.show()
    fi.show()
    ft.show()

    if SAVE_FIGS:
        stem = sect.replace(" ", "_")
        if fu is not None:
            fu.write_html(FIG_DIR / f"{stem}_umap.html")
        fi.write_html(FIG_DIR / f"{stem}_index.html")
        ft.write_html(FIG_DIR / f"{stem}_te.html")



=== Basic Materials ===


/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.




=== Communication Services ===


/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.




=== Consumer Cyclical ===


/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



... (12 outputs left out)


=== Technology ===


/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.




=== Utilities ===


/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



# Chapter 2 — Results  
**Sector Proxy vs Stock Representativeness**

This chapter investigates whether sector exposure can be faithfully captured by a single representative stock, or whether diversification within the sector is required to control idiosyncratic risk. Rather than assuming sectors are homogeneous, we progressively build evidence—starting from internal structure, moving to return behavior, and ending with risk-based validation—to answer this question in a data-driven way.

---

## 2.1 Intra-Sector Structure: How Cohesive Are Sectors?

We begin by examining the internal organization of each sector using UMAP embeddings and hierarchical clustering based on rolling correlation features. This step reveals whether a sector behaves as a single coherent group or whether it contains distinct sub-behaviors.

In most sectors, stocks collapse into one dominant cluster, indicating strong common dynamics across constituents. From a structural perspective, this suggests that simplified sector representation may be feasible. Financial Services stands out as an exception: the presence of multiple clusters highlights meaningful internal segmentation, immediately signaling that a single stock may struggle to capture the full sector behavior.

This structural view sets the stage for what follows. It tells us where simplification is even plausible—and where diversification is structurally justified before any performance comparison is made.

---

## 2.2 From Structure to Candidates: Which Stocks Look Representative?

Building on this structural analysis, we compute rolling correlations between each stock and its sector proxy over 60-day and 120-day windows. Averaging these correlations over time yields a representativeness score, allowing us to rank stocks within each sector.

Across sectors, only a small subset of stocks consistently displays high correlation to the proxy. Importantly, this group does not always coincide with the largest or most visible names in the sector. This highlights a key insight: representativeness is a behavioral property, not a size effect.

At this stage, the question shifts from *whether* a sector can be simplified to *whether the most representative candidates actually succeed in tracking the sector over time*.

---

## 2.3 Does the Representative Stock Really Track the Sector?

To answer this, we compare normalized indices (base = 100) for the sector proxy, the top-ranked representative stock, and a Top-5 basket constructed from the highest representativeness scores.

In some sectors—most notably Utilities (XEL) and Financial Services (JPM)—the representative stock closely follows the sector proxy over the entire period. For an investor, this means that holding a single well-chosen stock would have delivered a return path very similar to that of the sector itself.

In other sectors, including Consumer Cyclical, Consumer Defensive, Industrials, and Basic Materials, the representative stock tracks the proxy on average but exhibits visible deviations during certain periods. These divergences hint at residual idiosyncratic risk that a single stock cannot fully eliminate.

The contrast is sharpest in Real Estate (KIM) and Healthcare (PFE), where the representative stock persistently diverges from the proxy. In these cases, visual inspection alone suggests that a one-stock solution fails to capture sector dynamics.

---

## 2.4 How Concentrated Are Sector Movements?

To understand why some sectors diverge more than others, we examine concentration risk using Top-N contribution shares. This analysis measures how much of the sector’s daily movement is driven by its largest contributors.

Some sectors are clearly dominated by a small number of stocks, with the Top-1 or Top-3 names accounting for a large fraction of daily returns. In such environments, relying on a single stock significantly increases exposure to firm-specific shocks. Other sectors display broader participation, where returns are more evenly distributed across constituents, making simplified exposure more defensible.

This concentration lens explains why sectors with similar correlation structures can behave very differently once real-world risk is considered.

---

## 2.5 Tracking Error: When Simplification Breaks Down

We finally quantify representativeness using rolling tracking error between the representative stock and the sector proxy, and compare it to the tracking error of a Top-5 basket.

Utilities and Financial Services exhibit consistently low and stable tracking error for the single stock, confirming that simplification works not only on average but also through time. In Technology, Industrials, and Basic Materials, tracking error remains moderate, yet the Top-5 basket clearly improves stability by smoothing deviations.

Consumer Cyclical and Consumer Defensive sectors show frequent tracking-error spikes, often mitigated by diversification. Real Estate and Healthcare stand out again, with high and unstable tracking error even for the best candidate stock, reinforcing the conclusion that intra-sector diversification is essential.

Across nearly all sectors, the basket reduces tracking error, particularly during volatile or transitional periods—precisely when investors care most about reliable exposure.

---

## 2.6 Sector-Level Takeaways

| Sector Type | Conclusion |
|------------|-----------|
| Single stock sufficient | Utilities (XEL), Financial Services (JPM) |
| Single stock acceptable, basket safer | Technology (AAPL), Industrials (GE), Basic Materials (FCX) |
| Diversification adds clear value | Consumer Cyclical (F), Consumer Defensive (MNST) |
| Single stock not sufficient | Real Estate (KIM), Healthcare (PFE) |

---

## 2.7 What This Means for Investors

Sector investing can be simplified—but only selectively. Structural homogeneity makes simplification possible, but it does not guarantee robustness. True representativeness must be confirmed by stable tracking and low tracking error across market regimes.

A small, carefully chosen basket—often one stock per behavioral cluster—captures most of the diversification benefit while keeping complexity low. In short, some sectors can be owned with one stock, while others require diversification not because of theory, but because the data leaves no alternative.
